# Fashion MNIST 모델 개선 실험: 은닉층 추가에서 시작하기

은닉층이 하나인 기준 모델에서 출발해, 은닉층 추가와 하이퍼파라미터 조정이 분류 성능을 어떻게 바꾸는지 비교합니다.

**코드 표기**
- `[기준 코드]`: 참고 교재(「혼자 공부하는 머신러닝+딥러닝」 7장)의 코드를 그대로 또는 거의 그대로 쓴 부분
- `추가` / `변경`: 이 실험을 위해 새로 작성하거나 바꾼 부분

**실험 설계**: 단계마다 한 가지 조건만 바꾸고, 그 단계에서 가장 좋은 설정을 다음 단계로 넘깁니다.

| 단계 | 실험 | 바꾸는 조건 |
|---|---|---|
| 1. 은닉층 추가 | A → B | 기준 모델에 은닉층(50) 추가 |
| 2. 추가 층 dropout | C | 추가한 은닉층에 dropout 0.3 |
| 3. 유닛 수 확장 | D | 100-50 → 256-128 |
| 4. dropout 비율 | E, F | 2차 층 dropout 0.3 → 0.2 → 0.1 |
| 5. 옵티마이저 종류 | G, H | Adam ↔ RMSprop, SGD(네스테로프) |
| 6. Adam 학습률 | I, J, K | 0.001 → 0.0005 → 0.0003 → 0.0002 |
| 7. 활성화 함수 | L, M, N | ReLU ↔ ELU, Swish, GELU |

실행하면 학습 기록과 그림이 `results/` 폴더에 저장됩니다.


In [ ]:
# [기준 코드] 실행마다 동일한 결과를 얻기 위한 랜덤 시드 고정
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.random.set_seed(42)
tf.config.experimental.enable_op_determinism()

In [ ]:
# 추가: 그래프 한글 폰트 설정 + 결과 저장 폴더 준비
import os, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

OUT = 'results'                 # 체크포인트, 학습 기록, 그림을 모두 이 폴더에 저장
os.makedirs(OUT, exist_ok=True)

In [ ]:
# [기준 코드] 데이터 적재 + 0~1 정규화 + 훈련/검증 분리
from sklearn.model_selection import train_test_split

(train_input, train_target), (test_input, test_target) = \
    keras.datasets.fashion_mnist.load_data()

train_scaled = train_input / 255.0

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

# 추가: 최종 모델의 일반화 성능 확인용 테스트셋 정규화 (모델 선택에는 사용하지 않음, 마지막 셀에서만 사용)
test_scaled = test_input / 255.0

In [ ]:
# [기준 코드] 기준 모델 함수 model_fn: 은닉층 1개(100, relu) + 선택적 층 + 출력층 (참고용, 실험은 아래 build_model 사용)
def model_fn(a_layer=None):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(28,28)))
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(100, activation='relu'))
    if a_layer:
        model.add(a_layer)
    model.add(keras.layers.Dense(10, activation='softmax'))
    return model


# 추가: 은닉층 개수/유닛 수/드롭아웃 비율/활성화 함수를 바꿔가며 실험하기 위한 확장 함수
#   hidden = [(유닛수, 드롭아웃비율), ...]  -> 리스트 길이 = 은닉층 개수
#   드롭아웃 비율이 0이면 해당 은닉층 뒤에 Dropout을 붙이지 않음
#   예) model_fn(keras.layers.Dropout(0.3)) (기준 모델) = build_model([(100, 0.3)])
def build_model(hidden, act='relu'):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(28,28)))
    model.add(keras.layers.Flatten())
    for units, rate in hidden:                       # 추가: 은닉층 반복 추가
        model.add(keras.layers.Dense(units, activation=act))
        if rate > 0:
            model.add(keras.layers.Dropout(rate))   # 추가: 은닉층 뒤 dropout
    model.add(keras.layers.Dense(10, activation='softmax'))
    return model


# 추가: 옵티마이저 생성 함수 (lr=None이면 각 옵티마이저의 기본 학습률 사용, 7-2)
def make_optimizer(opt='adam', lr=None):
    if opt == 'adam':
        return keras.optimizers.Adam(learning_rate=lr or 0.001)
    if opt == 'rmsprop':
        return keras.optimizers.RMSprop(learning_rate=lr or 0.001)
    if opt == 'sgd_nesterov':
        return keras.optimizers.SGD(learning_rate=lr or 0.01, momentum=0.9, nesterov=True)
    raise ValueError(f'알 수 없는 옵티마이저: {opt}')

In [ ]:
# 추가: 실험 목록 (단계마다 한 가지 조건만 바꾸고, 그 단계에서 가장 좋은 설정을 다음 단계로 넘김)
#   1단계   은닉층 추가       : A(기준 모델) -> B(은닉층 50 추가)
#   2단계  추가 층 dropout   : C(추가한 은닉층 뒤에 dropout 0.3)
#   3단계 유닛 수 확장      : D(100-50 -> 256-128)
#   4단계 dropout 비율 조정 : E(2차 0.2), F(2차 0.1)
#   5단계 옵티마이저 종류   : G(RMSprop), H(SGD 네스테로프)  ※ 각 옵티마이저의 기본 학습률 사용
#   6단계 Adam 학습률       : I(0.0005), J(0.0003), K(0.0002)
#   7단계 활성화 함수       : L(ELU), M(Swish), N(GELU)
H2 = [(256, 0.3), (128, 0.2)]       # 4단계에서 고른 은닉층 설정 (= E)

experiments = {
    'A. 기준 모델(100+drop0.3)': dict(stage='1. 은닉층 추가',     hidden=[(100,0.3)],           act='relu',  opt='adam',         lr=0.001),
    'B. A+은닉층(100-50)':      dict(stage='1. 은닉층 추가',     hidden=[(100,0.3),(50,0.0)],  act='relu',  opt='adam',         lr=0.001),
    'C. B+추가층 drop0.3':      dict(stage='2. 추가 층 dropout', hidden=[(100,0.3),(50,0.3)],  act='relu',  opt='adam',         lr=0.001),
    'D. C의 유닛 256-128':      dict(stage='3. 유닛 수 확장',    hidden=[(256,0.3),(128,0.3)], act='relu',  opt='adam',         lr=0.001),
    'E. D+2차 drop0.2':         dict(stage='4. dropout 비율',    hidden=[(256,0.3),(128,0.2)], act='relu',  opt='adam',         lr=0.001),
    'F. D+2차 drop0.1':         dict(stage='4. dropout 비율',    hidden=[(256,0.3),(128,0.1)], act='relu',  opt='adam',         lr=0.001),
    'G. E+RMSprop':             dict(stage='5. 옵티마이저',      hidden=H2,                    act='relu',  opt='rmsprop',      lr=None),
    'H. E+SGD 네스테로프':      dict(stage='5. 옵티마이저',      hidden=H2,                    act='relu',  opt='sgd_nesterov', lr=None),
    'I. E+학습률0.0005':        dict(stage='6. Adam 학습률',     hidden=H2,                    act='relu',  opt='adam',         lr=0.0005),
    'J. E+학습률0.0003':        dict(stage='6. Adam 학습률',     hidden=H2,                    act='relu',  opt='adam',         lr=0.0003),
    'K. E+학습률0.0002':        dict(stage='6. Adam 학습률',     hidden=H2,                    act='relu',  opt='adam',         lr=0.0002),
    'L. K+ELU':                 dict(stage='7. 활성화 함수',     hidden=H2,                    act='elu',   opt='adam',         lr=0.0002),
    'M. K+Swish':               dict(stage='7. 활성화 함수',     hidden=H2,                    act='swish', opt='adam',         lr=0.0002),
    'N. K+GELU':                dict(stage='7. 활성화 함수',     hidden=H2,                    act='gelu',  opt='adam',         lr=0.0002),
}

In [ ]:
# 추가: 실험 실행
#   학습 기록을 results/histories.json에 실험 하나가 끝날 때마다 저장
#   -> 커널이 꺼져도 다시 학습할 필요 없음. 이름과 설정(hidden, act, opt, lr)이 같은 실험은 저장된 결과를 사용하고,
#      설정이 바뀐 실험만 새로 학습함
CACHE = os.path.join(OUT, 'histories.json')
KEYS = ('hidden', 'act', 'opt', 'lr')

def cfg_key(cfg):
    return json.loads(json.dumps({k: cfg[k] for k in KEYS}))   # 튜플/리스트 차이를 없애고 비교

cache = {}
if os.path.exists(CACHE):
    with open(CACHE, encoding='utf-8') as f:
        cache = json.load(f)

histories, results = {}, {}
for name, cfg in experiments.items():
    c = cache.get(name)
    if c and c['cfg'] == cfg_key(cfg) and os.path.exists(c['result']['ckpt']):
        histories[name], results[name] = c['history'], dict(c['result'])
        tag = '(저장된 결과 사용)'
    else:
        print(f'학습 중: {name} ...', flush=True)
        keras.utils.set_random_seed(42)                       # 모든 실험을 같은 초기 조건에서 시작
        model = build_model(cfg['hidden'], cfg['act'])

        model.compile(optimizer=make_optimizer(cfg['opt'], cfg['lr']),        # 변경: 실험별 옵티마이저
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])                                     # [기준 코드]

        ckpt_path = os.path.join(OUT, f'best-model-{name[0]}.keras')          # 실험 알파벳으로 파일 이름
        checkpoint_cb = keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True)          # [기준 코드]
        early_stopping_cb = keras.callbacks.EarlyStopping(patience=3,                            # 변경: 2 -> 3
                                                          restore_best_weights=True)             # [기준 코드]

        history = model.fit(train_scaled, train_target, epochs=50, verbose=0,                   # 변경: 20 -> 50
                            validation_data=(val_scaled, val_target),
                            callbacks=[checkpoint_cb, early_stopping_cb])                       # [기준 코드]

        val_loss, val_acc = model.evaluate(val_scaled, val_target, verbose=0)   # 가장 좋았던 가중치로 평가
        h = {k: [float(v) for v in vals] for k, vals in history.history.items()}
        histories[name] = h
        results[name] = dict(ckpt=ckpt_path, val_loss=float(val_loss), val_acc=float(val_acc),
                             best_epoch=int(np.argmin(h['val_loss'])) + 1, total_epochs=len(h['loss']),
                             params=int(model.count_params()))
        cache[name] = dict(cfg=cfg_key(cfg), history=h, result=results[name])
        with open(CACHE, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False)
        tag = ''
    results[name]['stage'] = cfg['stage']

    r = results[name]
    pad = ' ' * max(0, 26 - sum(2 if ord(ch) > 127 else 1 for ch in name))   # 한글 폭 맞춤 출력
    print(f"{name}{pad} | val_loss {r['val_loss']:.4f} | val_acc {r['val_acc']:.4f} "
          f"| 최적 epoch {r['best_epoch']} / 실행 {r['total_epochs']} {tag}")

In [ ]:
# 결과 1 : 실험별 성능 비교표 (실험 순서대로) -> results/fig1_results_table.png
df = pd.DataFrame(results).T[['stage','params','best_epoch','total_epochs','val_loss','val_acc']]
df = df.astype({'params':int,'best_epoch':int,'total_epochs':int,'val_loss':float,'val_acc':float})
df['val_acc(%)'] = (df['val_acc']*100).round(2)
df = df.round({'val_loss':4,'val_acc':4})
display(df)
df.to_csv(os.path.join(OUT, 'results.csv'), encoding='utf-8-sig')

best_name = min(results, key=lambda k: results[k]['val_loss'])   # 검증 손실이 가장 낮은 실험 선택
print('\n▶ 최종 선택 모델:', best_name)

# 표 그림
stage_best = {s: g['val_loss'].idxmin() for s, g in df.groupby('stage')}   # 단계별 최저 검증 손실
cols = ['단계', '실험', '파라미터 수', '최적 epoch', '실행 epoch', '검증 손실', '검증 정확도(%)']
rows = [[r['stage'], n, f"{r['params']:,}", r['best_epoch'], r['total_epochs'],
         f"{r['val_loss']:.4f}", f"{r['val_acc(%)']:.2f}"] for n, r in df.iterrows()]
fig, ax = plt.subplots(figsize=(11, 0.36 * (len(rows) + 1)), dpi=200)
ax.axis('off')
tbl = ax.table(cellText=rows, colLabels=cols, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.35)
tbl.auto_set_column_width(col=list(range(len(cols))))
for (i, j), cell in tbl.get_celld().items():
    cell.set_edgecolor('#d1d5db')
    if i == 0:
        cell.set_facecolor('#e5e7eb'); cell.set_text_props(weight='bold')
    elif rows[i-1][1] == best_name:
        cell.set_facecolor('#fde2e2'); cell.set_text_props(weight='bold')   # 최종 모델
    elif stage_best.get(rows[i-1][0]) == rows[i-1][1]:
        cell.set_facecolor('#eef6ff')                                        # 각 단계의 최고 모델
plt.savefig(os.path.join(OUT, 'fig1_results_table.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 결과 2 : 모든 실험의 epoch에 따른 훈련/검증 손실 그래프 -> results/fig2_loss_all.png
n = len(histories)
ncol = 4
nrow = (n + ncol - 1) // ncol
fig, axs = plt.subplots(nrow, ncol, figsize=(16, 3.4 * nrow), dpi=150)
axs = axs.ravel()

for ax, (name, h) in zip(axs, histories.items()):
    epochs = range(1, len(h['loss']) + 1)                 # epoch을 1부터 표시
    best_ep = int(np.argmin(h['val_loss'])) + 1
    ax.plot(epochs, h['loss'], label='Train', color='#1f77b4', linewidth=1.5)
    ax.plot(epochs, h['val_loss'], label='Val', color='#ff7f0e', linewidth=1.5)
    ax.axvline(best_ep, color='gray', linestyle='--', linewidth=1, label=f'Best ({best_ep})')
    ax.set_title(name, fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('Epoch', fontsize=8); ax.set_ylabel('Loss', fontsize=8)
    ax.tick_params(labelsize=8)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(fontsize=7, loc='upper right')

for ax in axs[n:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig2_loss_all.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 결과 3 : 단계별 최고 모델의 검증 손실 비교 -> results/fig3_stage_best.png
#   각 단계에서 검증 손실이 가장 낮은 모델 하나씩 (단계가 진행될수록 어떻게 바뀌는지)
stage_best = {}
for name in experiments:
    s = results[name]['stage']
    if s not in stage_best or results[name]['val_loss'] < results[stage_best[s]]['val_loss']:
        stage_best[s] = name

plt.figure(figsize=(9, 5), dpi=150)
cmap = plt.get_cmap('viridis', len(stage_best) + 1)
for i, (s, name) in enumerate(stage_best.items()):
    h = histories[name]
    is_final = (name == best_name)
    plt.plot(range(1, len(h['val_loss']) + 1), h['val_loss'],
             label=f"{s} → {name} (min {min(h['val_loss']):.4f})",
             color='#d62728' if is_final else cmap(i), linewidth=2.6 if is_final else 1.4)
plt.title('단계별 최고 모델의 검증 손실 변화', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Validation Loss')
plt.legend(fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig3_stage_best.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 결과 4 : 최종 선택 모델의 손실/정확도 그래프 (epoch에 따른 성능 변화) -> results/fig4_final_curves.png
h = histories[best_name]
epochs = range(1, len(h['loss']) + 1)
best_ep = int(np.argmin(h['val_loss'])) + 1
fig, axs = plt.subplots(1, 2, figsize=(12, 4), dpi=150)

axs[0].plot(epochs, h['loss'], label='train'); axs[0].plot(epochs, h['val_loss'], label='val')     # [기준 코드] 손실 그래프
axs[0].set_xlabel('epoch'); axs[0].set_ylabel('loss'); axs[0].set_title(f'{best_name} - loss')

axs[1].plot(epochs, h['accuracy'], label='train'); axs[1].plot(epochs, h['val_accuracy'], label='val')
axs[1].set_xlabel('epoch'); axs[1].set_ylabel('accuracy'); axs[1].set_title(f'{best_name} - accuracy')

for ax in axs:
    ax.axvline(best_ep, color='gray', linestyle='--', linewidth=1, label=f'best ({best_ep})')
    ax.grid(True, linestyle=':', alpha=0.6); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig4_final_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 결과 5 : 저장된 최상의 모델을 다시 불러와 구조와 검증/테스트 성능 확인 -> results/fig5_final_model.png
best_model = keras.models.load_model(results[best_name]['ckpt'])      # [기준 코드] load_model
best_model.save(os.path.join(OUT, 'best-model.keras'))                 # 최종 모델 저장
best_model.summary()                                                   # 모델 구조 (은닉층 추가 확인용)

val_res = best_model.evaluate(val_scaled, val_target, verbose=0)       # [loss, accuracy]
test_res = best_model.evaluate(test_scaled, test_target, verbose=0)    # 추가: 테스트 세트는 여기서 한 번만 사용
print('검증 세트 :', val_res)
print('테스트 세트:', test_res)

# 추가: 기준 모델(A)도 같은 방법으로 평가해 최종 모델과 비교
base_name = list(experiments)[0]
base_model = keras.models.load_model(results[base_name]['ckpt'])
base_val = base_model.evaluate(val_scaled, val_target, verbose=0)
base_test = base_model.evaluate(test_scaled, test_target, verbose=0)
cmp = pd.DataFrame({'검증 손실': [base_val[0], val_res[0]], '검증 정확도(%)': [base_val[1]*100, val_res[1]*100],
                    '테스트 손실': [base_test[0], test_res[0]], '테스트 정확도(%)': [base_test[1]*100, test_res[1]*100],
                    '학습 파라미터 수': [base_model.count_params(), best_model.count_params()]},
                   index=[base_name, best_name]).round(4)
display(cmp)

# 그림: 층 구조 표 + 기준 모델과의 비교 표
def out_shape(layer):
    try:
        return str(tuple(layer.output.shape))
    except Exception:
        return '-'
layer_rows = [[l.name, l.__class__.__name__, out_shape(l), f'{l.count_params():,}'] for l in best_model.layers]
layer_rows.append(['합계 (학습 파라미터)', '', '', f'{best_model.count_params():,}'])
cmp_rows = [[i, f"{r['검증 손실']:.4f}", f"{r['검증 정확도(%)']:.2f}", f"{r['테스트 손실']:.4f}",
             f"{r['테스트 정확도(%)']:.2f}", f"{int(r['학습 파라미터 수']):,}"] for i, r in cmp.iterrows()]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 0.3 * (len(layer_rows) + len(cmp_rows) + 4)), dpi=200,
                             gridspec_kw={'height_ratios': [len(layer_rows) + 1, len(cmp_rows) + 1]})
for ax, title, cols_, rows_ in [
        (a1, f'최종 모델 구조: {best_name}', ['Layer', 'Type', 'Output Shape', 'Param #'], layer_rows),
        (a2, '기준 모델(A)과 최종 모델 비교', ['모델', '검증 손실', '검증 정확도(%)', '테스트 손실', '테스트 정확도(%)', '파라미터 수'], cmp_rows)]:
    ax.axis('off'); ax.set_title(title, fontsize=11, fontweight='bold', loc='left')
    t = ax.table(cellText=rows_, colLabels=cols_, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.3)
    t.auto_set_column_width(col=list(range(len(cols_))))
    for (i, j), cell in t.get_celld().items():
        cell.set_edgecolor('#d1d5db')
        if i == 0:
            cell.set_facecolor('#e5e7eb'); cell.set_text_props(weight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig5_final_model.png'), dpi=300, bbox_inches='tight')
plt.show()